# pbncalib_botsort — Kaggle pipeline

Option A field registration (PnLCalib + temporal smoothing) inside the SoccerNet
GSR pipeline, with an A/B against the BroadTrack baseline.

**Four stages, each gating the next.** Run them in order and stop at the first
failure — a later stage's output is meaningless if an earlier one did not pass.

| Stage | What | Gates |
|---|---|---|
| 1 | build env (uv + Python 3.9) → `preflight_imports.py` | every stage imports |
| 2 | `verify_pnlcalib_env.py` | **answers the torch question** |
| 3 | shortest sequence end to end | `bbox_pitch` populated and trustworthy |
| 4 | GS-HOTA A/B vs BroadTrack | the actual verdict |

### The environment build is not trivial

Kaggle's session Python is past 3.9 and ships torch 2.x. This pipeline needs
Python 3.9 and torch 1.13.1 — the detector, reid and tracking stages require it.
So Stage 1 installs `uv`, has it fetch a 3.9 interpreter, and builds a separate
venv: the work `preflight_cpu.sh` does elsewhere. **Expect a slow first cell.**

Do **not** `pip install -r optiona_sfr/requirements.txt` into the session. It
would try to move the session's torch and can leave CUDA mismatched against the
driver; the file's own header says so. Everything below installs into the venv
with `--python .venv`, never into the session.

## Stage 1 — environment

In [ ]:
import subprocess
from pathlib import Path

REPO_URL = "https://github.com/Yass1223/pbncalib_botsort.git"
WORK = Path("/kaggle/working")
REPO = WORK / "pbncalib_botsort"
VENV = REPO / ".venv"


def sh(cmd, cwd=None, check=True):
    print("+", cmd, flush=True)
    r = subprocess.run(cmd, shell=True, cwd=cwd)
    if check and r.returncode:
        raise RuntimeError("failed (%d): %s" % (r.returncode, cmd))
    return r.returncode


if not (REPO / "pyproject.toml").exists():
    sh("git clone --depth 1 %s %s" % (REPO_URL, REPO))
else:
    print("repo already present")

In [ ]:
# uv, a Python 3.9 interpreter, and the venv.
sh("pip -q install uv", check=False)
sh("uv python install 3.9")
sh("uv venv --python 3.9 %s" % VENV, cwd=REPO)

# Resolve fresh from pyproject.toml. uv.lock is intentionally absent: shapely,
# matplotlib and scipy became direct dependencies of the calibration stage and
# the old lock predates them, so syncing it would build an environment in which
# the stage fails at import.
sh("uv pip install --python %s -e ." % VENV, cwd=REPO)

PY = "uv run --python %s python" % VENV
sh(PY + " -c \"import torch, numpy; print('venv torch', torch.__version__, "
        "'| numpy', numpy.__version__)\"", cwd=REPO)

In [ ]:
# GATE 1: every _target_ declared under configs/ must import.
rc = sh(PY + " scripts/preflight_imports.py", cwd=REPO, check=False)
assert rc == 0, ("Stage 1 failed: some pipeline stages do not import. Fix the "
                 "dependency first -- later stages cannot be interpreted if the "
                 "pipeline is only half-built.")
print("\nStage 1 PASSED")

## Stage 2 — does PnLCalib run on torch 1.13.1?

**This is the gate the whole integration hangs on.** PnLCalib pins torch 2.3.1;
the pipeline pins 1.13.1 and cannot move. The models use only long-stable ops,
but two things cannot be settled by reading source:

- can torch 1.13.1 read a checkpoint archive written by 2.3.1? (check **D**)
- do the `state_dict` keys match the architecture from `hrnetv2_w48.yaml`? (check **E**)

If D or E fail, **do not bump the pipeline's torch.** The remedy — a subprocess
boundary, or re-serialised checkpoints — lands inside the single
`compute_cameras` function in `optiona_api.py`, which exists to contain exactly
this.

In [ ]:
# ~505 MiB of checkpoints. Resume-safe, so re-running is cheap.
sh("bash scripts/setup_pnlcalib.sh", cwd=REPO)

In [ ]:
FRAME = next(iter(sorted(Path("/kaggle/input").rglob("img1/000001.jpg"))), None)
print("frame:", FRAME)
assert FRAME is not None, "no SoccerNet frame found under /kaggle/input"

rc = sh(PY + " scripts/verify_pnlcalib_env.py"
        " --repo pretrained_models/pnlcalib/PnLCalib"
        " --weights-kp pretrained_models/pnlcalib/weights/SV_kp"
        " --weights-line pretrained_models/pnlcalib/weights/SV_lines"
        " --frame %s" % FRAME, cwd=REPO, check=False)
if rc != 0:
    raise SystemExit(
        "Stage 2 FAILED. Read which check failed above.\n"
        "  D (torch.load) -> 1.13.1 cannot read the 2.3.1 archive.\n"
        "  E (state_dict) -> keys/shapes disagree with hrnetv2_w48.yaml.\n"
        "Either way the fix goes inside compute_cameras() in optiona_api.py. "
        "DO NOT bump the pipeline's torch: the detector, reid and tracking "
        "stages require 1.13.1.")
print("\nStage 2 PASSED -- PnLCalib runs on the pipeline's pins")

## Stage 3 — one sequence, end to end

Shortest available sequence, so a failure surfaces in minutes rather than after
a whole split. The two local gates run first: they need no GPU, and the
conversion gate carries the negative control that proves it can fail.

In [ ]:
# GATE 3a: camera-conversion parity + negative control.
rc = sh(PY + " scripts/verify_optiona_conversion.py", cwd=REPO, check=False)
assert rc == 0, "conversion parity failed -- bbox_pitch would be wrong everywhere"

# GATE 3b: engine contract (pure pandas, no torch).
rc = sh(PY + " tests/test_optiona_api_contract.py", cwd=REPO, check=False)
assert rc == 0, "engine contract broken"

In [ ]:
roots = sorted(Path("/kaggle/input").rglob("*/img1"))
assert roots, "no SoccerNet-GSR sequences found under /kaggle/input"
seqs = sorted((len(list(r.glob("*.jpg"))), r.parent) for r in roots)
n_frames, SEQ = seqs[0]
DATA_ROOT = SEQ.parent.parent
print("shortest sequence: %s (%d frames)" % (SEQ.name, n_frames))
print("dataset root:", DATA_ROOT)

In [ ]:
sh(PY + " -m tracklab.main -cn soccernet_optiona"
        " dataset.dataset_path=%s dataset.nvid=1"
        " 'dataset.vids_dict.test=[%s]'"
        " experiment_name=optiona_%s" % (DATA_ROOT, SEQ.name, SEQ.name),
   cwd=REPO)

### Read the result — completeness is not evidence

Completeness is ~1.0 by construction: smoothing interpolates gaps, carry-forward
fills the rest, and the tracker never returns `None` after lock-on. A camera that
locks on at frame 1, drifts, and is carried for the remaining frames gives zero
errors, 100% completeness and wrong coordinates everywhere. These are the numbers
that separate that case from a working one.

In [ ]:
import json
import numpy as np

calib = sorted((REPO / "optiona_calib").glob("*.json"))
assert calib, "no calibration JSON written -- the stage did not run"
payload = json.loads(calib[0].read_text())
frames = payload["frames"]
names = sorted(frames)

s = np.array([frames[n]["s"] for n in names], float)
fin = s[np.isfinite(s)]
print("sequence %s (%d calibrated frames)\n" % (payload["sequence"], len(names)))
print("s ACROSS THE WHOLE SEQUENCE, not just at lock-on:")
print("  min %.3f   p10 %.3f   median %.3f   p90 %.3f   max %.3f"
      % (fin.min(), np.percentile(fin, 10), np.median(fin),
         np.percentile(fin, 90), fin.max()))
print("  frames below 0.5: %d/%d" % (int((fin < 0.5).sum()), len(fin)))
first = next((n for n in names if np.isfinite(frames[n]["s"])), None)
print("  first lock-on: %s" % first)
print("  NOTE 0.5 is a STARTING HEURISTIC borrowed from BroadTrack's ONLINE")
print("       reinit rule, not a calibrated cut for this post-smoothing score.")

q = len(fin) // 4
if q:
    print("\n  quartile medians: " +
          "  ".join("%.3f" % np.median(fin[i * q:(i + 1) * q]) for i in range(4)))
    print("  a monotonically falling trend = locked on, then drifted")

P = [frames[n]["parameters"] for n in names]
f_ = np.array([p["x_focal_length"] for p in P])
pos = np.array([p["position_meters"] for p in P])
print("\ncamera parameters:")
print("  focal  %8.1f .. %8.1f   (spread %.1f%% of mean)"
      % (f_.min(), f_.max(), (f_.max() - f_.min()) / f_.mean() * 100))
for i, ax in enumerate("xyz"):
    print("  pos %s  %8.2f .. %8.2f m" % (ax, pos[:, i].min(), pos[:, i].max()))
travel = float(np.sum(np.linalg.norm(np.diff(pos, axis=0), axis=1)))
print("  focal-point travel: %.1f m" % travel)
print("  (NBJW travels up to 20 m per sequence; large travel means the")
print("   focal/distance degeneracy is unchecked)")

In [ ]:
# Out-of-bounds count, straight from the stage's own log line.
# LOWER BOUND, not a measurement: a behind-camera unprojection can land inside
# the bounds by coincidence, so 0% means "none caught", not "none present".
sh("grep -h 'projected positions' outputs/optiona_%s/*/*/*.log 2>/dev/null "
   "|| grep -rh 'projected positions' outputs/ 2>/dev/null | tail -5"
   % SEQ.name, cwd=REPO, check=False)

## Stage 4 — GS-HOTA A/B vs BroadTrack

The only number that settles whether Option A is better. Every other stage is
frozen, so the delta is attributable to calibration alone.

In [ ]:
bt_state = REPO / "states" / ("broadtrack_%s.pklz" % SEQ.name)
if not bt_state.exists():
    sh("bash scripts/setup_broadtrack.sh", cwd=REPO, check=False)
    sh(PY + " -m tracklab.main -cn soccernet"
            " dataset.dataset_path=%s dataset.nvid=1"
            " 'dataset.vids_dict.test=[%s]'"
            " experiment_name=broadtrack_%s" % (DATA_ROOT, SEQ.name, SEQ.name),
       cwd=REPO, check=False)
else:
    print("using cached BroadTrack run:", bt_state)

In [ ]:
import re

for tag in ("optiona_%s" % SEQ.name, "broadtrack_%s" % SEQ.name):
    d = REPO / "outputs" / tag
    print("\n=== %s ===" % tag)
    if not d.exists():
        print("  no output directory -- did the run complete?")
        continue
    found = False
    for h in sorted(d.rglob("*.json"))[-5:]:
        try:
            obj = json.loads(h.read_text())
        except Exception:
            continue
        if not isinstance(obj, dict):
            continue
        for k, v in obj.items():
            if re.search(r"hota|deta|assa|accuracy", str(k), re.I):
                print("  %s: %s" % (k, v))
                found = True
    if not found:
        print("  no HOTA-like keys found; check the run log for the eval table")

## What "ready" means

Not "it ran". Ready is:

- **parity + negative control** from Stage 3a, with the control failing loudly
- **`s` across the whole sequence**, with no falling quartile trend
- **out-of-bounds count** at or near zero — remembering it is a lower bound
- **camera parameters physically plausible** — focal not drifting, optical
  centre not wandering off the touchline
- **GS-HOTA at least matching BroadTrack** on identical sequences

If any is missing, the run is not evidence — regardless of completeness.